In [1]:
import sys

In [2]:
print(sys.executable)

c:\Users\divya\Desktop\AI\LangChain_Lab\.venv\Scripts\python.exe


In [3]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from common.llm import get_llm
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage, SystemMessage

In [4]:
@tool
def calculator(expression: str) -> str:
    """Evaluate a basic math expression, e.g. '235 * 18' or '(10 + 5) / 3'.
    Only use this for arithmetic calculations."""
    try:
        # eval() runs the string as Python code -> "235 * 8" becomes actual multiplication
        result = eval(expression, {"__builtins__": {}}) # restricted eval, no dangerous builtins
        return str(result)
    except Exception as e:
        return f"Error evaluating expression: {e}"


In [5]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a given city name."""
    # Mock data — real project mein yahan ek actual weather API call hoga (jaise OpenWeatherMap)
    fake_weather_db = {
        "kolkata": "28°C, light rain",
        "delhi": "34°C, clear sky",
        "mumbai": "30°C, humid",
    }
    city_key = city.lower()
    return fake_weather_db.get(city_key, f"No weather data available for {city}")

In [6]:
tools = [calculator, get_weather]

In [10]:
tools_by_name = {t.name: t for t in tools}
print(tools_by_name)

{'calculator': StructuredTool(name='calculator', description="Evaluate a basic math expression, e.g. '235 * 18' or '(10 + 5) / 3'.\nOnly use this for arithmetic calculations.", args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x000002BBDB9696C0>), 'get_weather': StructuredTool(name='get_weather', description='Get the current weather for a given city name.', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x000002BBDB969940>)}


In [11]:
llm = get_llm(temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [20]:
response = llm_with_tools.invoke("What is 235 * 18?")

print(response.content)        # ye khali/blank ho sakta hai
print(response.tool_calls)     # yahan model ka decision dikhega


[{'name': 'calculator', 'args': {'expression': '235 * 18'}, 'id': 'fc_e18dd8f0-f851-431d-8e6e-0423737d76fb', 'type': 'tool_call'}]


In [13]:
def run_agent(user_input: str, max_iterations: int = 5):
    # Conversation history yahin maintain hogi (Project 2 jaisa concept, bas manual)
    messages = [
        SystemMessage(content="You are a helpful assistant. Use tools when needed, otherwise answer directly."),
        HumanMessage(content=user_input),
    ]

    for step in range(max_iterations):
        # Step 1: model se pucho - tool chahiye ya seedha jawab de sakta hai
        ai_response = llm_with_tools.invoke(messages)
        messages.append(ai_response)   # model ka response history mein add karo

        # Step 2: check karo - model ne koi tool maanga?
        if not ai_response.tool_calls:
            # Koi tool call nahi -> matlab ye final answer hai, loop rok do
            return ai_response.content

        # Step 3: model ne tool(s) maange -> unhe chalao
        for tool_call in ai_response.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_id = tool_call["id"]

            print(f"[Agent decided] calling '{tool_name}' with args {tool_args}")

            # Sahi tool function dhundo aur usse chalao
            selected_tool = tools_by_name[tool_name]
            tool_result = selected_tool.invoke(tool_args)

            # Tool ka result wapas conversation history mein daalo,
            # taaki model agli baar isse dekh ke aage decide kar sake
            messages.append(ToolMessage(content=str(tool_result), tool_call_id=tool_id))

    return "Max iterations reached without a final answer."

In [14]:
print(run_agent("What is 235 * 18?"))

[Agent decided] calling 'calculator' with args {'expression': '235 * 18'}
235 × 18 = **4,230**


In [15]:
print(run_agent("What's the weather like in Kolkata right now?"))

[Agent decided] calling 'get_weather' with args {'city': 'Kolkata'}
In Kolkata right now it’s about **28 °C** with **light rain**.


In [16]:
print(run_agent("What is the capital of France?"))

The capital of France is **Paris**.


In [17]:
print(run_agent("What is 50 * 3, and also what's the weather in Delhi?"))

[Agent decided] calling 'get_weather' with args {'city': 'Delhi'}
- **50 × 3 = 150**  
- **Weather in Delhi:** 34 °C, clear sky


In [18]:
messages = [
    SystemMessage(content="You are a helpful assistant. Use tools when needed, otherwise answer directly."),
    HumanMessage(content="What is 100 + 250?"),
]

for step in range(5):
    ai_response = llm_with_tools.invoke(messages)
    messages.append(ai_response)
    if not ai_response.tool_calls:
        break
    for tool_call in ai_response.tool_calls:
        result = tools_by_name[tool_call["name"]].invoke(tool_call["args"])
        messages.append(ToolMessage(content=str(result), tool_call_id=tool_call["id"]))

for m in messages:
    print(f"[{m.type}] {getattr(m, 'content', '')}")

[system] You are a helpful assistant. Use tools when needed, otherwise answer directly.
[human] What is 100 + 250?
[ai] 100 + 250 = 350.
